# SIMCA one-class classification — Wine dataset demo

SIMCA(Soft Independent Modeling of
Class Analogy) fits a PCA model on samples from a single **target class**,
then classifies new samples by how far they deviate from that model — using
the **Q** (residual outside the PCA subspace) and **T2** (Hotelling distance
inside the subspace) statistics. No labeled examples of other classes are
needed to build the model; they're only used here to measure how well it
rejects them.

This notebook runs the same workflow three times on the
Wine dataset (Forina et al.)
(178 wines, 13 physicochemical measurements, 3 cultivars), each time varying
one thing, to show how the pieces fit together:

| Block | Train/test split | `tuning` | Target class |
|---|---|---|---|
| 1 | none (all data as "train") | `rigorous` | `class_0` |
| 2 | held-out test set | `rigorous` | `class_1` |
| 3 | held-out test set | `compliant` | `class_2` |

Block 1 shows every diagnostic plot the library has, plus how `OCCSplit`
actually divides the data for cross-validation. Blocks 2 and 3 stay lean —
metrics and the acceptance-boundary plot only — since the point there is the
split/tuning difference, not another plot tour.


In [23]:
import sys, os

try:
    from OneClassClassifier import SIMCA, OCCSplit, SCORING, plot_searchcv
except ImportError:
    # Package not installed (no `pip install -e .` yet) -- fall back to
    # importing straight from the repo, assuming this notebook is running
    # from inside the OneClassClassifier folder itself.
    sys.path.insert(0, os.path.dirname(os.getcwd()))
    from OneClassClassifier import SIMCA, OCCSplit, SCORING, plot_searchcv

import numpy as np
import pandas as pd

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)
pd.set_option("display.precision", 3)


## Data

Wine's features span very different scales (e.g. `proline` is in the
thousands, `nonflavanoid_phenols` is below 1) — PCA is scale-sensitive, so a
`StandardScaler` is used as the model's `preprocessor` in every block. It's
fitted only on target-class samples, matching how SIMCA is meant to be used.


In [24]:
wine = load_wine()
X, y = wine.data, wine.target
feature_names = [str(f) for f in wine.feature_names]
class_names = [str(c) for c in wine.target_names]

print(f"X shape: {X.shape}")
print(f"Classes: {dict(zip(class_names, np.bincount(y)))}")

pd.DataFrame(X, columns=feature_names).assign(target=[class_names[i] for i in y]).head()


X shape: (178, 13)
Classes: {'class_0': np.int64(59), 'class_1': np.int64(71), 'class_2': np.int64(48)}


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,class_0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,class_0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,class_0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,class_0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,class_0


---
## Block 1 — no split, `tuning="rigorous"`, target = `class_0`

All 178 samples are used as "train" — there's no held-out test set, so the
final metrics below are computed on the same data the model was tuned on
(optimistic, not a fair generalization estimate). This block exists to show
the full diagnostic toolkit, not to report honest performance — that's what
blocks 2 and 3 are for.


In [25]:
TARGET_1 = 0  # class_0

cv1 = OCCSplit(target_class=TARGET_1, cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE))


### How OCCSplit divides the data

In [26]:
for fold, (train_idx, test_idx) in enumerate(cv1.split(X, y)):
    n_target_train = int(np.sum(y[train_idx] == TARGET_1))
    n_nontarget_train = int(np.sum(y[train_idx] != TARGET_1))
    n_target_test = int(np.sum(y[test_idx] == TARGET_1))
    n_nontarget_test = int(np.sum(y[test_idx] != TARGET_1))
    print(
        f"fold {fold}: train={len(train_idx):3d} (target={n_target_train}, non-target={n_nontarget_train})"
        f" | test={len(test_idx):3d} (target={n_target_test}, non-target={n_nontarget_test})"
    )


fold 0: train= 47 (target=47, non-target=0) | test=131 (target=12, non-target=119)
fold 1: train= 47 (target=47, non-target=0) | test=131 (target=12, non-target=119)
fold 2: train= 47 (target=47, non-target=0) | test=131 (target=12, non-target=119)
fold 3: train= 47 (target=47, non-target=0) | test=131 (target=12, non-target=119)
fold 4: train= 48 (target=48, non-target=0) | test=130 (target=11, non-target=119)


train_idx is target rows only (nothing else ever reaches `fit()`); test_idx
carries the held-out target rows *plus* every non-target row, on every
single fold — so specificity is always evaluated against the complete
non-target set, not a random slice of it.

### Why `scoring=SCORING` is here

If you don't pass `scoring` to `GridSearchCV` at all, it falls back to
calling the estimator's own `score(X, y)` — both to pick the winning
configuration *and* as the single number it reports back in `cv_results_`
(`mean_test_score`). That default works, but the number it gives you isn't
always something you'd want to read directly:

- Under `tuning="rigorous"`, `score()` returns an internal selection
  criterion (rewards the most complex model whose sensitivity clears the
  nominal `1 - alpha` level) — not a percentage, not meant to be read on
  its own.
- Under `tuning="compliant"`, `score()` *is* already the efficiency value,
  so the default alone would actually be readable there — but you'd still
  only get that one combined number, not sensitivity and specificity
  broken out separately.

`SCORING` (exported by the library) is a ready-made `scoring` dict —
`{"objective", "sensitivity", "specificity", "efficiency"}` — that fixes
both problems in one line: `"objective"` is a scorer that just calls
`estimator.score(X, y)`, so it reproduces the exact selection behaviour you'd
get by not passing `scoring` at all, and the other three entries compute
real, readable numbers via `score_metrics()` instead. It has to be a dict
(not a single function) because that's the only way to get more than one
column back in `cv_results_`; and once `scoring` is a dict, scikit-learn
requires you to say explicitly which entry decides the winner — that's what
`refit="objective"` is for. Without it, `GridSearchCV` has no way to know
which of the four numbers should govern `best_params_`.

Under `tuning="rigorous"` specificity/efficiency come back as `NaN` — there's
nothing for `score_metrics()` to compute them from in that mode — and
scikit-learn prints a harmless warning about it, expected here.


In [27]:
grid_search_1 = GridSearchCV(
    SIMCA(method="alt", alpha=0.05, preprocessor=StandardScaler(), target_class=TARGET_1, tuning="rigorous"),
    param_grid={"n_components": list(range(1, 9))},
    cv=cv1,
    scoring=SCORING,     # ready-made {objective, sensitivity, specificity, efficiency} scorers
    refit="objective",   # picks the winner the same way score() alone would
)
grid_search_1.fit(X, y)
print(f"Best params: {grid_search_1.best_params_}")


Best params: {'n_components': 7}


c:\Users\alber\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning:

One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan]



In [28]:
cv_results_1 = pd.DataFrame(grid_search_1.cv_results_)
cv_results_1[["param_n_components", "mean_test_sensitivity", "std_test_sensitivity"]] \
    .sort_values("param_n_components").reset_index(drop=True)


,param_n_components,mean_test_sensitivity,std_test_sensitivity
0,1,0.932,0.034
1,2,0.932,0.063
2,3,0.932,0.063
3,4,0.932,0.063
4,5,0.932,0.063
5,6,0.932,0.063
6,7,0.932,0.097
7,8,0.829,0.121


In [29]:
plot_searchcv(grid_search_1, x_param="n_components", y_metric="sensitivity")


In [30]:
model_1 = grid_search_1.best_estimator_
print(f"Selected model: n_components={model_1.n_components}, method={model_1.method!r}")

metrics_1 = model_1.metrics(X, y)
print("Metrics (same data used for tuning -- optimistic):")
for name, value in metrics_1.items():
    if isinstance(value, dict):
        print(f"  {name}:")
        for cls, val in value.items():
            print(f"    {class_names[cls]}: {val:.3f}")
    elif isinstance(value, float):
        print(f"  {name}: {value:.3f}")
    else:
        print(f"  {name}: {value}")


Selected model: n_components=7, method='alt'
Metrics (same data used for tuning -- optimistic):
  sensitivity: 1.000
  specificity: 0.966
  specificity_by_class:
    class_1: 0.944
    class_2: 1.000
  efficiency: 0.983
  FAR: 0.034
  FRR: 0.000
  n_target: 59
  n_non_target: 119


### Acceptance boundary (T2 vs Q)

In [31]:
model_1.plot_t2q(X, [class_names[i] for i in y], color_by="label",log_scale=True)


### PCA scores

In [32]:
model_1.plot_scores(X, [class_names[i] for i in y])


### Loadings

In [33]:
model_1.plot_loadings(mode="line", variable_names=feature_names)


In [34]:
model_1.plot_scores_loadings(X, [class_names[i] for i in y], variable_names=feature_names)


### Scree plot

In [35]:
model_1.plot_scree()


### Contribution plot

Which variables push a rejected sample's Q statistic the most — picking the
first non-target sample that got rejected.


In [36]:
y_pred_1 = model_1.predict(X)
rejected_1 = np.flatnonzero((y != TARGET_1) & (y_pred_1 == model_1.negative_label))
flagged_1 = X[[rejected_1[0]]]
print(f"Sample class: {class_names[y[rejected_1[0]]]!r}")

model_1.plot_contributions(flagged_1, kind="q", variable_names=feature_names)


Sample class: 'class_1'


### Confusion matrix

In [37]:
model_1.plot_confusion_matrix(y, y_pred_1, show_percent=True)


---
## Train / test split for blocks 2 and 3

Stratified across all three classes, shared by both remaining blocks so
they're evaluated on the same held-out data.


In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape[0]} samples - {dict(zip(class_names, np.bincount(y_train)))}")
print(f"Test:  {X_test.shape[0]} samples - {dict(zip(class_names, np.bincount(y_test)))}")


Train: 124 samples - {'class_0': np.int64(41), 'class_1': np.int64(50), 'class_2': np.int64(33)}
Test:  54 samples - {'class_0': np.int64(18), 'class_1': np.int64(21), 'class_2': np.int64(15)}


---
## Block 2 — held-out test set, `tuning="rigorous"`, target = `class_1`


In [39]:
TARGET_2 = 1  # class_1
cv2 = OCCSplit(target_class=TARGET_2, cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE))

grid_search_2 = GridSearchCV(
    SIMCA(method="alt", alpha=0.05, preprocessor=StandardScaler(), target_class=TARGET_2, tuning="rigorous"),
    param_grid={"n_components": list(range(1, 9))},
    cv=cv2,
    scoring=SCORING,
    refit="objective",
)
grid_search_2.fit(X_train, y_train)
print(f"Best params: {grid_search_2.best_params_}")


Best params: {'n_components': 2}


c:\Users\alber\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning:

One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan]



In [40]:
model_2 = grid_search_2.best_estimator_
print(f"Selected model: n_components={model_2.n_components}, method={model_2.method!r}")

metrics_2 = model_2.metrics(X_test, y_test)
print("Test set metrics:")
for name, value in metrics_2.items():
    if isinstance(value, dict):
        print(f"  {name}:")
        for cls, val in value.items():
            print(f"    {class_names[cls]}: {val:.3f}")
    elif isinstance(value, float):
        print(f"  {name}: {value:.3f}")
    else:
        print(f"  {name}: {value}")


Selected model: n_components=2, method='alt'
Test set metrics:
  sensitivity: 0.857
  specificity: 0.636
  specificity_by_class:
    class_0: 0.556
    class_2: 0.733
  efficiency: 0.739
  FAR: 0.364
  FRR: 0.143
  n_target: 21
  n_non_target: 33


In [41]:
model_2.plot_t2q(X_test, [class_names[i] for i in y_test], color_by="label")


---
## Block 3 — held-out test set, `tuning="compliant"`, target = `class_2`

Same `SCORING` dict as the other blocks — under `tuning="compliant"` every
entry in it (sensitivity, specificity, efficiency) comes back as a real
number, no `NaN` columns this time.


In [42]:
TARGET_3 = 2  # class_2
cv3 = OCCSplit(target_class=TARGET_3, cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE))

grid_search_3 = GridSearchCV(
    SIMCA(method="alt", alpha=0.05, preprocessor=StandardScaler(), target_class=TARGET_3, tuning="compliant"),
    param_grid={"n_components": list(range(1, 9))},
    cv=cv3,
    scoring=SCORING,
    refit="objective",
)
grid_search_3.fit(X_train, y_train)
print(f"Best params: {grid_search_3.best_params_}")


Best params: {'n_components': 2}


In [43]:
model_3 = grid_search_3.best_estimator_
print(f"Selected model: n_components={model_3.n_components}, method={model_3.method!r}")

metrics_3 = model_3.metrics(X_test, y_test)
print("Test set metrics:")
for name, value in metrics_3.items():
    if isinstance(value, dict):
        print(f"  {name}:")
        for cls, val in value.items():
            print(f"    {class_names[cls]}: {val:.3f}")
    elif isinstance(value, float):
        print(f"  {name}: {value:.3f}")
    else:
        print(f"  {name}: {value}")


Selected model: n_components=2, method='alt'
Test set metrics:
  sensitivity: 1.000
  specificity: 0.974
  specificity_by_class:
    class_0: 1.000
    class_1: 0.952
  efficiency: 0.987
  FAR: 0.026
  FRR: 0.000
  n_target: 15
  n_non_target: 39


In [44]:
model_3.plot_t2q(X_test, [class_names[i] for i in y_test], color_by="label")
